# Part I. ETL Pipeline for Pre-Processing the Files

## PLEASE RUN THE FOLLOWING CODE FOR PRE-PROCESSING THE FILES

#### Import Python packages

In [ ]:
# Import Python packages
import pandas as pd
import cassandra
import re
import os
import glob
import numpy as np
import json
import csv

#### Creating list of filepaths to process original event csv data files

In [ ]:
# checking your current working directory
print(os.getcwd())

# Get your current folder and subfolder event data
filepath = os.getcwd() + '/event_data'

# Create a for loop to create a list of files and collect each filepath
file_path_list = []
for root, dirs, files in os.walk(filepath):
    file_path_list.extend(glob.glob(os.path.join(root, '*.csv')))

print('CSV files discovered:', len(file_path_list))

#### Processing the files to create the data file csv that will be used for Apache Cassandra tables

In [ ]:
# initiating an empty list of rows that will be generated from each file
full_data_rows_list = []

# for every filepath in the file path list
for f in sorted(file_path_list):
    # reading csv file
    with open(f, 'r', encoding='utf8', newline='') as csvfile:
        # creating a csv reader object
        csvreader = csv.reader(csvfile)
        next(csvreader)

        # extracting each data row one by one and append it
        for line in csvreader:
            full_data_rows_list.append(line)

# creating a smaller event data csv file called event_datafile_new.csv that will be used to insert data into the
# Apache Cassandra tables
csv.register_dialect('myDialect', quoting=csv.QUOTE_ALL, skipinitialspace=True)

with open('event_datafile_new.csv', 'w', encoding='utf8', newline='') as f:
    writer = csv.writer(f, dialect='myDialect')
    writer.writerow(['artist','firstName','gender','itemInSession','lastName','length',
                'level','location','sessionId','song','userId'])
    for row in full_data_rows_list:
        if (row[0] == ''):
            continue
        writer.writerow((row[0], row[2], row[3], row[4], row[5], row[6], row[7], row[8], row[12], row[13], row[16]))

In [ ]:
# check the number of rows in your csv file
with open('event_datafile_new.csv', 'r', encoding='utf8') as f:
    print(sum(1 for line in f))

# Part II. Complete the Apache Cassandra coding portion of your project.

The `event_datafile_new.csv` file contains these columns:
- artist
- firstName of user
- gender of user
- item number in session
- last name of user
- length of the song
- level (paid or free song)
- location of the user
- sessionId
- song title
- userId

## Begin writing your Apache Cassandra code in the cells below

#### Creating a Cluster

In [ ]:
# This should make a connection to a Cassandra instance your local machine (127.0.0.1)
from cassandra.cluster import Cluster
cluster = Cluster(['127.0.0.1'])

# To establish connection and begin executing queries, need a session
session = cluster.connect()

#### Create Keyspace

In [ ]:
# Create a Keyspace
session.execute("""
CREATE KEYSPACE IF NOT EXISTS sparkify
WITH REPLICATION = {
    'class': 'SimpleStrategy',
    'replication_factor': 1
}
""")

#### Set Keyspace

In [ ]:
# Set KEYSPACE to the keyspace specified above
session.set_keyspace('sparkify')

### Now we need to create tables to run the following queries.

With Apache Cassandra, the database is modeled around the queries. This solution creates one table for each analytics question.

## Create queries to ask the following three questions of the data

### 1. Give me the artist, song title and song length in the music app history that was heard during `sessionId = 338`, and `itemInSession = 4`

### 2. Give me only the following: artist name, song title sorted by `itemInSession`, and user first and last name for `userId = 10`, `sessionId = 182`

### 3. Give me every user first and last name in my music app history who listened to the song `All Hands Against His Own`

## Query 1 table: songplays_by_session_item

In [ ]:
## Query 1: artist, song title, and length by sessionId and itemInSession
query = """
CREATE TABLE IF NOT EXISTS songplays_by_session_item (
    session_id int,
    item_in_session int,
    artist text,
    song text,
    length double,
    PRIMARY KEY (session_id, item_in_session)
)
"""
session.execute(query)

In [ ]:
file = 'event_datafile_new.csv'

query = """
INSERT INTO songplays_by_session_item
(session_id, item_in_session, artist, song, length)
VALUES (%s, %s, %s, %s, %s)
"""

with open(file, encoding='utf8') as f:
    csvreader = csv.reader(f)
    next(csvreader) # skip header
    for line in csvreader:
        session.execute(query, (int(line[8]), int(line[3]), line[0], line[9], float(line[5])))

#### Do a SELECT to verify that the data have been inserted into the table

In [ ]:
query = """
SELECT artist, song, length
FROM songplays_by_session_item
WHERE session_id = 338 AND item_in_session = 4
"""
rows = session.execute(query)
for row in rows:
    print(row.artist, row.song, row.length)

## Query 2 table: songplays_by_user_session

In [ ]:
## Query 2: artist, song, and user by userId and sessionId, sorted by itemInSession
query = """
CREATE TABLE IF NOT EXISTS songplays_by_user_session (
    user_id int,
    session_id int,
    item_in_session int,
    artist text,
    song text,
    first_name text,
    last_name text,
    PRIMARY KEY ((user_id, session_id), item_in_session)
) WITH CLUSTERING ORDER BY (item_in_session ASC)
"""
session.execute(query)

In [ ]:
query = """
INSERT INTO songplays_by_user_session
(user_id, session_id, item_in_session, artist, song, first_name, last_name)
VALUES (%s, %s, %s, %s, %s, %s, %s)
"""

with open(file, encoding='utf8') as f:
    csvreader = csv.reader(f)
    next(csvreader) # skip header
    for line in csvreader:
        session.execute(query, (int(line[10]), int(line[8]), int(line[3]), line[0], line[9], line[1], line[4]))

In [ ]:
query = """
SELECT artist, song, first_name, last_name
FROM songplays_by_user_session
WHERE user_id = 10 AND session_id = 182
"""
rows = session.execute(query)
for row in rows:
    print(row.artist, row.song, row.first_name, row.last_name)

## Query 3 table: users_by_song

In [ ]:
## Query 3: users who listened to a specific song
query = """
CREATE TABLE IF NOT EXISTS users_by_song (
    song text,
    user_id int,
    first_name text,
    last_name text,
    PRIMARY KEY (song, user_id)
)
"""
session.execute(query)

In [ ]:
query = """
INSERT INTO users_by_song
(song, user_id, first_name, last_name)
VALUES (%s, %s, %s, %s)
"""

with open(file, encoding='utf8') as f:
    csvreader = csv.reader(f)
    next(csvreader) # skip header
    for line in csvreader:
        session.execute(query, (line[9], int(line[10]), line[1], line[4]))

In [ ]:
query = """
SELECT first_name, last_name
FROM users_by_song
WHERE song = 'All Hands Against His Own'
"""
rows = session.execute(query)
for row in rows:
    print(row.first_name, row.last_name)

### Drop the tables before closing out the sessions

In [ ]:
session.execute('DROP TABLE IF EXISTS songplays_by_session_item')
session.execute('DROP TABLE IF EXISTS songplays_by_user_session')
session.execute('DROP TABLE IF EXISTS users_by_song')

### Close the session and cluster connection

In [ ]:
session.shutdown()
cluster.shutdown()